# ClarifySQL · Notebook 01 — Compute pilot

**Goal.** Before we fix data sizes and the training plan, measure what one free T4 GPU can really do
with `Qwen/Qwen3-4B-Instruct-2507` (proposal, Section 13: *expected fit, then measured*).

**Time:** about 20–30 minutes; most of it is the one-time 8 GB model download.
**GPU quota used:** about half an hour.

**Before you press *Run all***
- **Kaggle:** in the right-hand panel open *Session options* → **Accelerator: GPU T4 x2**, and turn
  **Internet: On** (needs a phone-verified account).
- **Colab:** *Runtime → Change runtime type →* **T4 GPU**.

**At the end**, download `results/pilot/pilot_metrics.json` (last cell explains how) and copy it into the
repository folder `results/pilot/` on your laptop.

## 0. Check the GPU

`nvidia-smi` lists the GPUs this notebook can use. On Kaggle you should see **two Tesla T4** cards with
about 15 GB each. They are separate cards: one model must fit on *one* of them. If you see
*"command not found"* or no GPU, the accelerator is not switched on (see the first cell).

In [ ]:
!nvidia-smi

## 1. Get the project code

This cell clones the project from GitHub (branch `main`) into the notebook's working folder and makes
the `clarifysql` package importable. Running it again later just pulls the newest commits.

*Private repository?* Add a secret called `GITHUB_TOKEN` (Kaggle: **Add-ons → Secrets**; Colab: the
**key icon** in the left bar) and set `USE_TOKEN = True`. Our repository is public, so this is not needed.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Sathviktr/LMA_ClarifySQL.git"
BRANCH = "main"
USE_TOKEN = False  # only needed for a private repository

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
REPO = f"{WORK}/LMA_ClarifySQL"

if USE_TOKEN:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    else:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{token}@")

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

sys.path.insert(0, f"{REPO}/src")
print("Running on:", "Kaggle" if ON_KAGGLE else "Colab / other", "| repo at", REPO)

In [ ]:
# Libraries that are not preinstalled (or too old) on Kaggle/Colab. PyTorch with CUDA is already there.
!pip install -q -r {REPO}/requirements-gpu.txt

## 2. Environment report

Records the versions of Python, PyTorch, CUDA and the Hugging Face libraries, plus each GPU's memory and
*compute capability*. The T4 has capability 7.5, which means **no native BF16**. That is why we compute
in FP16.

In [ ]:
from clarifysql import pilot, config

results = {"model_id": config.MODEL_ID, "env": pilot.env_report()}
results["env"]

## 3. FP16: load the full model in 16-bit

4.02 billion parameters × 2 bytes ≈ **8 GB** of weights. The first run downloads them from Hugging Face
(a few minutes). `peak_allocated_gb` is the most memory PyTorch used at any moment.

In [ ]:
model, tok, seconds = pilot.load_model(config.MODEL_ID, mode="fp16")
results["fp16_load"] = {"seconds": round(seconds, 1), **pilot.gpu_memory()}
results["fp16_load"]

## 4. FP16 sanity check

The weights were published in BF16. FP16 has a much smaller range (largest number 65,504), so a value can
overflow to `inf`. We run one forward pass, check that every output number (*logit*) is finite, and look
at a real greedy answer to a tiny text-to-SQL question.

In [ ]:
results["fp16_sanity"] = pilot.fp16_sanity(model, tok)
print("logits finite:", results["fp16_sanity"]["logits_finite"])
print(results["fp16_sanity"]["sample_answer"])

## 5. Generation speed

To produce **one** token the GPU must read **all** 8 GB of weights. With a batch of 4 sequences it produces
4 tokens per read, so throughput should rise almost 4×. We also time one *policy-sized* call: about 600
prompt tokens in and 60 tokens (one JSON action) out. This is the unit for estimating the cost of the
agent's episodes.

In [ ]:
results["fp16_generation"] = pilot.measure_generation(model, tok, batch_sizes=(1, 4), max_new_tokens=128)
results["fp16_call_latency"] = pilot.measure_call_latency(model, tok)
results["fp16_generation"], results["fp16_call_latency"]

## 6. Free the GPU before loading the 4-bit version

In [ ]:
del model
pilot.free_gpu()
pilot.gpu_memory()

## 7. NF4: load the model in 4-bit (how QLoRA stores the frozen weights)

Each weight becomes a 4-bit code plus a shared scale per block of 64 weights, so we expect roughly
**2.5–3 GB**. Generation in 4-bit is usually *slower*, because every weight must be unpacked before use.
That is why the proposal runs inference in FP16.

In [ ]:
model4, tok, seconds = pilot.load_model(config.MODEL_ID, mode="nf4")
results["nf4_load"] = {"seconds": round(seconds, 1), **pilot.gpu_memory()}
results["nf4_generation"] = pilot.measure_generation(model4, tok, batch_sizes=(1,), max_new_tokens=64)
results["nf4_load"], results["nf4_generation"]

## 8. QLoRA training steps

We attach a LoRA adapter (rank 16 on the 7 attention and MLP matrices of every layer) and time 4 training
steps at **1,024** and **2,048** tokens with micro-batch 1 and gradient checkpointing. The input is random
token ids, so the loss means nothing here. We are measuring memory and speed. The trainable-parameter
count should equal the formula: **33,030,144**.

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained(config.MODEL_ID).to_dict()
results["lora_formula_params"] = pilot.lora_param_count(cfg, r=16)
results["qlora"], peft_model = pilot.measure_qlora_steps(model4, seq_lens=(1024, 2048), steps=4)
results["qlora"]

## 9. Save and summarise

In [ ]:
import datetime

results["finished_at"] = datetime.datetime.now().isoformat(timespec="seconds")
path = pilot.save_results(results, f"{WORK}/results/pilot/pilot_metrics.json")
print("saved", path, "\n")
print(pilot.summarize(results))

## 10. Send the results back

- **Kaggle (interactive session):** in the right-hand panel open **Output** (or the file browser) →
  `/kaggle/working/results/pilot/pilot_metrics.json` → **Download**.
  If you used **Save Version → Save & Run All**, open that version → **Output** tab → download the file.
- **Colab:** click the **folder icon** in the left bar → `/content/results/pilot/pilot_metrics.json` →
  right-click → **Download**.

Copy the file to `LMA_ClarifySQL/results/pilot/pilot_metrics.json` on your laptop and tell Claude the
pilot is done. Then **stop the session** (Kaggle: power icon / *Stop session*), so it does not use up GPU
quota while idle.